# Classification — Stroke Prediction

> **The same work as the report, in a notebook.** Nothing here is new: every cell below is the
> code of `code/`, as it is there, in the order `main.py` runs it, one cell per definition. Running the whole
> notebook writes the same figures to `figures/` and the same tables and numbers to `results/` as
> `python code/main.py` does. The report itself is [`index.md`](index.md), next to this notebook.

One section per step of the report. Steps 0 to 4 use only the training rows and their validation folds; steps 5
and 6 are the only ones that receive the test rows.

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

# The notebook runs from anywhere inside the repository and uses the same paths as the scripts.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mkdocs.yml").is_file())
CODE_DIR = REPO / "docs" / "projects" / "classification" / "code"


def show_figures(*numbers):
    """Show the figures the scripts just wrote to figures/."""
    for n in numbers:
        display(Image(filename=str(FIGURES_DIR / f"fig{n}.png")))


def show_tables(step):
    """Show the tables of one step, as the report prints them."""
    display(Markdown((RESULTS_DIR / f"{step}_tables.md").read_text()))

## [`../eda/code/preprocessing.py`](../eda/code/preprocessing.py) — the EDA's data, split and pipeline, reused unchanged

In [ ]:
"""Data loading, the train/test split and the preprocessing pipeline of the Stroke EDA.

Importable: nothing is fitted at import time. Typical use from the modeling deliverable:

    from preprocessing import load_raw, split, build_preprocessor
    X_train, X_test, y_train, y_test = split(load_raw())
    preprocess = build_preprocessor()
    X_train_t = preprocess.fit_transform(X_train)   # fit on train only
    X_test_t = preprocess.transform(X_test)         # test is only transformed

Every choice in build_preprocessor() is motivated in section 4A of the report.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

SEED = 42
DATA_PATH = REPO / "data" / "stroke-prediction" / "healthcare-dataset-stroke-data.csv"

TARGET = "stroke"
ID_COLUMN = "id"
SKEWED_FEATURES = ["avg_glucose_level", "bmi"]      # right-skewed (skewness > 1): log1p, then standardize
SYMMETRIC_FEATURES = ["age"]                         # near-symmetric: standardize only
NUMERICAL_FEATURES = SYMMETRIC_FEATURES + SKEWED_FEATURES
CATEGORICAL_FEATURES = ["gender", "hypertension", "heart_disease", "ever_married",
                        "work_type", "Residence_type", "smoking_status"]
FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

### `load_raw`

In [ ]:
def load_raw(path=DATA_PATH):
    """Read the CSV exactly as published. The text 'N/A' in bmi is parsed as NaN by pandas."""
    return pd.read_csv(path)

### `split`

In [ ]:
def split(df, test_size=0.2):
    """Stratified 80/20 split with a fixed seed; returns X_train, X_test, y_train, y_test.

    The file is sorted by the target (every positive comes first), so the split must shuffle;
    stratification keeps the minority rate the same in both parts.
    """
    X = df[FEATURES]
    y = df[TARGET]
    return train_test_split(X, y, test_size=test_size, stratify=y, shuffle=True, random_state=SEED)

### `build_preprocessor`

In [ ]:
def build_preprocessor():
    """Return an unfitted ColumnTransformer whose output is a dense, finite matrix.

    - age: median imputation (a safeguard: the column has no missing value) + standardization;
    - avg_glucose_level, bmi: median imputation + log1p + standardization;
    - bmi missing indicator: 0/1, kept unscaled;
    - categorical: constant imputation + full one-hot (no dropped level, so no category is
      placed closer to the others than they are to each other); a category never seen in
      training is encoded as an all-zero block, so it is never confused with a known one.
    """
    symmetric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    skewed = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", symmetric, SYMMETRIC_FEATURES),
        ("log", skewed, SKEWED_FEATURES),
        ("missing", MissingIndicator(features="all"), ["bmi"]),
        ("cat", categorical, CATEGORICAL_FEATURES),
    ], remainder="drop")

## [`code/common.py`](code/common.py) — the settings fixed in step 0, the folds and the helpers

In [ ]:
"""Settings fixed before training, the EDA's data and pipeline, the cross-validation folds and shared helpers.

The data, the train/test split and the preprocessing pipeline are imported unchanged from the EDA's code
(docs/projects/eda/code/preprocessing.py): this deliverable starts from that pipeline, as the statement requires.
"""

import json
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # render straight to files, so the scripts also run without a display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

HERE = CODE_DIR
REPORT = HERE.parent / "index.md"
FIGURES_DIR = HERE.parent / "figures"
RESULTS_DIR = HERE.parent / "results"
EDA_CODE = HERE.parents[1] / "eda" / "code"
EDA_SPLIT = HERE.parents[1] / "eda" / "results" / "split.csv"

# Appended after this folder: `common` stays this file, and `preprocessing` is the EDA's module. The EDA's own
# common.py cannot be imported under the same name, so the small report helpers below repeat it.

__all__ = ["HERE", "REPORT", "FIGURES_DIR", "RESULTS_DIR", "FEATURES", "SEED", "build_preprocessor", "MAIN_METRIC",
           "CRITERION_GAIN", "COST_MISS", "N_FOLDS", "SEEDS", "CLASS_COLORS", "MODEL_COLORS", "NEUTRAL",
           "load_raw", "load_data", "folds", "save_figure", "md_table", "Results"]

# --- Step 0: fixed before any model was trained --------------------------------------------------
MAIN_METRIC = "average precision"   # the decision of step 0, computed with sklearn's average_precision_score;
#                                     at a 4.87% prevalence it is read against the AP of a random ranking, 0.0487
CRITERION_GAIN = 0.15   # the network's mean test AP (over SEEDS) must be at least 15% above the best baseline's,
#                         and the gap must be larger than the network's standard deviation across SEEDS
COST_MISS = 20          # a missed stroke costs as much as 20 false alarms (a false alarm costs 1)
N_FOLDS = 5             # stratified, shuffled folds of the training rows: the validation sets
SEEDS = (42, 43, 44, 45, 46)

CLASS_COLORS = {0: "#2a78d6", 1: "#eb6834"}   # the EDA's colorblind-safe pair (blue / orange)
MODEL_COLORS = {"Network": "#eb6834", "Logistic regression": "#2a78d6", "Gradient boosting": "#1baf7a",
                "Majority class": "#898781"}
NEUTRAL = "#898781"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
    "axes.titlesize": 11, "legend.fontsize": 8, "legend.framealpha": 0.9,
})

### `load_data`

In [ ]:
def load_data():
    """The EDA's split, checked row by row against the one the EDA saved in results/split.csv."""
    X_train, X_test, y_train, y_test = split(load_raw())
    saved = pd.read_csv(EDA_SPLIT)
    for name, part in (("train", X_train), ("test", X_test)):
        if list(part.index) != saved.loc[saved["partition"] == name, "row"].tolist():
            raise ValueError(f"the {name} rows differ from the EDA's split")
    return X_train, X_test, y_train.to_numpy(), y_test.to_numpy()

### `folds`

In [ ]:
def folds(y_train):
    """The validation folds every model uses: N_FOLDS stratified, shuffled folds of the training rows."""
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    return list(splitter.split(np.zeros(len(y_train)), y_train))

### `save_figure`

In [ ]:
def save_figure(fig, number, title):
    """Give the figure its numbered title and write it to figures/fig{number}.png."""
    FIGURES_DIR.mkdir(exist_ok=True)
    fig.suptitle(f"Figure {number} — {title}", fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.3 / fig.get_figheight()))   # keep the panels clear of the title
    fig.savefig(FIGURES_DIR / f"fig{number}.png", bbox_inches="tight")
    plt.close(fig)

### `md_table`

In [ ]:
def md_table(df, floatfmt=None):
    """Render a DataFrame as a Markdown table; floatfmt maps a column to its format spec (default .4f).

    Integers get a thousands separator, negative numbers a true minus sign, as in the prose, and a missing value
    a dash.
    """
    floatfmt = floatfmt or {}
    header = "| " + " | ".join(str(c) for c in df.columns) + " |"
    rule = "|" + "|".join("---" for _ in df.columns) + "|"
    rows = []
    for values in df.astype(object).itertuples(index=False):   # object dtype keeps each cell's own type
        cells = []
        for c, v in zip(df.columns, values):
            if isinstance(v, (float, np.floating)) and np.isnan(v):
                text = "—"
            elif isinstance(v, (float, np.floating)):
                text = format(v, floatfmt.get(c, ".4f"))
            elif isinstance(v, (int, np.integer)) and not isinstance(v, (bool, np.bool_)):
                text = f"{v:,}"
            else:
                text = str(v)
            if isinstance(v, (int, float, np.number)) and text.startswith("-"):
                text = "−" + text[1:]
            cells.append(text)
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join([header, rule, *rows])

### `Results`

In [ ]:
class Results:
    """Collects the numbered tables and the numbers one step of the report quotes, then writes them to results/.

    Each table goes to results/table{NN}.csv and, rendered in Markdown, to results/{name}_tables.md; the numbers
    go to results/{name}_metrics.json. The table numbers are the ones the report uses.
    """

    def __init__(self, name):
        self.name = name
        self.metrics = {}
        self.tables = []
        RESULTS_DIR.mkdir(exist_ok=True)

    def table(self, number, title, df, floatfmt=None):
        """Register Table `number` and write its CSV."""
        self.tables.append(f"### Table {number} — {title}\n\n{md_table(df, floatfmt)}\n")
        df.to_csv(RESULTS_DIR / f"table{number:02d}.csv", index=False)

    def write(self):
        """Write the metrics JSON and the Markdown file with every table of this step."""
        (RESULTS_DIR / f"{self.name}_metrics.json").write_text(
            json.dumps(self.metrics, indent=2, default=float) + "\n")
        (RESULTS_DIR / f"{self.name}_tables.md").write_text("\n".join(self.tables))

## [`code/network.py`](code/network.py) — the network and its training loop

In [ ]:
"""The network of this deliverable: an MLP in PyTorch that returns one logit, and its training loop.

The training loop follows the statement's skeleton: one optimizer step per mini-batch, the validation loss after
every epoch, early stopping with a patience, and the best weights restored at the end (kept in memory rather
than in a file).
"""

import copy

import numpy as np
import torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 128     # about 6 positives per batch at the 4.87% rate; a batch without any is rare (0.17%)
MAX_EPOCHS = 300     # an upper bound; early stopping decides
PATIENCE = 30        # epochs without a lower validation loss before stopping; long on purpose, since a fold's
#                      validation loss moves from epoch to epoch with only 39 or 40 positives in it
DEFAULT = {"hidden": [32, 16], "dropout": 0.0, "lr": 1e-3, "weight_decay": 0.0}   # the network of step 2: a
#            starting point, not a tuned choice — step 4 searches architectures and learning rates around it

### `MLP`

In [ ]:
class MLP(nn.Module):
    """n_inputs -> hidden[0] -> ... -> hidden[-1] -> 1 logit, ReLU after every hidden layer and optional dropout
    after every ReLU. Hidden layers use He (Kaiming) normal initialization, the variance that keeps ReLU
    activations from shrinking or growing layer after layer; the output layer, which has no ReLU after it, uses
    Glorot (Xavier) normal. Every bias starts at 0."""

    def __init__(self, n_inputs, hidden, dropout=0.0):
        super().__init__()
        layers, width = [], n_inputs
        for units in hidden:
            linear = nn.Linear(width, units)
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            width = units
        output = nn.Linear(width, 1)
        nn.init.xavier_normal_(output.weight)
        nn.init.zeros_(output.bias)
        layers.append(output)
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        """The logit of a stroke, one per row: the loss applies the sigmoid itself."""
        return self.layers(x).squeeze(-1)

    def n_parameters(self):
        """The number of trainable weights and biases."""
        return sum(p.numel() for p in self.parameters())

### `to_tensor`

In [ ]:
def to_tensor(a):
    """A float32 tensor, the dtype of the network's weights."""
    return torch.as_tensor(np.asarray(a, dtype=np.float32))

### `predict_proba`

In [ ]:
def predict_proba(model, X):
    """Probability of a stroke for each row of X (the sigmoid of the logit), with dropout switched off."""
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(to_tensor(X))).numpy()

### `train`

In [ ]:
def train(X_train, y_train, config, seed, X_val=None, y_val=None, epochs=None):
    """Train one network. Returns (model, history, best_epoch).

    config: hidden (list of widths), dropout, lr, weight_decay. AdamW is used throughout; with weight_decay 0 it is
    exactly Adam. The decay applies to the weight matrices only: biases shift the activations rather than scale them,
    so they are left out, as the course recommends. The loss is BCEWithLogitsLoss with pos_weight = negatives /
    positives of y_train, so the few positives weigh as much in the gradient as all the negatives.

    With a validation set: up to MAX_EPOCHS epochs, early stopping on the validation loss with PATIENCE, and the
    weights of the best epoch restored. Without one (the final retraining): exactly `epochs` epochs.
    Both losses in the history are measured after each epoch with dropout off, so they are comparable.
    """
    if X_val is None and not epochs:
        raise ValueError("without a validation set to stop on, the number of epochs must be given")
    torch.manual_seed(seed)   # the initial weights and the dropout masks
    model = MLP(X_train.shape[1], config["hidden"], config["dropout"])
    y_train = np.asarray(y_train)
    pos_weight = torch.tensor((y_train == 0).sum() / (y_train == 1).sum(), dtype=torch.float32)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    weights = [p for p in model.parameters() if p.ndim > 1]    # the weight matrices
    biases = [p for p in model.parameters() if p.ndim == 1]
    optimizer = torch.optim.AdamW([{"params": weights, "weight_decay": config["weight_decay"]},
                                   {"params": biases, "weight_decay": 0.0}], lr=config["lr"])
    Xt, yt = to_tensor(X_train), to_tensor(y_train)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))   # the order of the mini-batches
    if X_val is not None:
        Xv, yv = to_tensor(X_val), to_tensor(y_val)

    history = {"train_loss": [], "val_loss": [], "val_ap": []}
    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(1, (epochs or MAX_EPOCHS) + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            history["train_loss"].append(loss_fn(model(Xt), yt).item())
            if X_val is None:
                continue
            val_logits = model(Xv)
            val_loss = loss_fn(val_logits, yv).item()
        history["val_loss"].append(val_loss)
        # AP depends only on the ranking, so the logits give the same value as the probabilities
        history["val_ap"].append(average_precision_score(y_val, val_logits.numpy()))
        if val_loss < best_loss:
            best_loss, best_epoch, wait = val_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())   # keep the best weights
        else:
            wait += 1
            if wait == PATIENCE:
                break
    if X_val is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = epochs
    return model, history, best_epoch

## [`code/validation.py`](code/validation.py) — the cross-validation every model shares

In [ ]:
"""Cross-validation on the training rows, shared by every model: the network and the baselines see exactly the same
fold matrices, built by a pipeline fitted on each fold's training part only."""

from typing import NamedTuple

import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

### `Fold`

In [ ]:
class Fold(NamedTuple):
    """One validation fold: the pipeline's output on the fold's training and validation parts, their labels, and the
    positions of the validation rows among the training rows."""
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    val_idx: np.ndarray

### `without_indicator`

In [ ]:
def without_indicator():
    """The EDA's pipeline without its BMI-missing indicator."""
    preprocessor = build_preprocessor()
    preprocessor.set_params(missing="drop")
    return preprocessor

### `fold_matrices`

In [ ]:
def fold_matrices(X_train, y_train, folds, make_preprocessor, prepare=None):
    """One Fold per validation fold. The pipeline is fitted on the fold's training part and only applied to its
    validation part. `prepare` optionally edits the raw rows first (the BMI sensitivity check)."""
    X = prepare(X_train) if prepare else X_train
    mats = []
    for train_idx, val_idx in folds:
        preprocessor = make_preprocessor()
        mats.append(Fold(preprocessor.fit_transform(X.iloc[train_idx]), y_train[train_idx],
                         preprocessor.transform(X.iloc[val_idx]), y_train[val_idx], val_idx))
    return mats

### `scores`

In [ ]:
def scores(y, p):
    """Average precision and ROC-AUC of the probabilities p for the labels y."""
    return average_precision_score(y, p), roc_auc_score(y, p)

### `cv_sklearn`

In [ ]:
def cv_sklearn(model, mats):
    """A scikit-learn baseline refitted on every fold; validation AP and ROC-AUC per fold and the out-of-fold
    probabilities (each training row predicted by the model that did not see it)."""
    out = {"ap": [], "auc": [], "oof": np.zeros(sum(len(fold.val_idx) for fold in mats))}
    for fold in mats:
        p = model.fit(fold.X_train, fold.y_train).predict_proba(fold.X_val)[:, 1]
        out["oof"][fold.val_idx] = p
        ap, auc = scores(fold.y_val, p)
        out["ap"].append(ap)
        out["auc"].append(auc)
    return out

### `cv_network`

In [ ]:
def cv_network(config, mats, seed=SEED):
    """One network per fold, early-stopped on the fold's validation part with its best weights restored; the same
    outputs as cv_sklearn, plus the best epochs and the curves."""
    out = {"ap": [], "auc": [], "oof": np.zeros(sum(len(fold.val_idx) for fold in mats)), "best_epoch": [],
           "histories": []}
    for fold in mats:
        model, history, best_epoch = train(fold.X_train, fold.y_train, config, seed, X_val=fold.X_val,
                                           y_val=fold.y_val)
        p = predict_proba(model, fold.X_val)
        out["oof"][fold.val_idx] = p
        ap, auc = scores(fold.y_val, p)
        out["ap"].append(ap)
        out["auc"].append(auc)
        out["best_epoch"].append(best_epoch)
        out["histories"].append(history)
    return out

### `mean_sd`

In [ ]:
def mean_sd(values):
    """Mean and sample standard deviation (ddof = 1) across folds or seeds."""
    return float(np.mean(values)), float(np.std(values, ddof=1))

# The data and the validation folds

The first lines of `main()`: the EDA's split, checked row by row, and the 5 stratified folds of the training
rows.

In [ ]:
X_train, X_test, y_train, y_test = load_data()
cv_folds = folds(y_train)
print(f"train {len(y_train)} rows ({int(y_train.sum())} strokes) · test {len(y_test)} rows "
      f"({int(y_test.sum())} strokes)")

## [`code/baselines.py`](code/baselines.py) — step 1: the trivial, linear and boosting baselines; step 0 compares them too, so they are defined first

In [ ]:
"""Step 1 — the baselines, on the same pipeline and the same validation folds as the network (Table 2)."""

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

### `make_baselines`

In [ ]:
def make_baselines():
    """The trivial and the linear baseline the statement requires, and gradient boosting as the optional one.
    The two learned ones weight the classes as the network's pos_weight does."""
    return {
        "Majority class": DummyClassifier(strategy="most_frequent"),
        "Logistic regression": LogisticRegression(class_weight="balanced", max_iter=1000),
        "Gradient boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
    }

### `baselines_run`

In [ ]:
def baselines_run(mats, y_train):
    """Cross-validate every baseline; returns the out-of-fold probabilities and the best baseline (highest mean AP)."""
    results = Results("baselines")
    rows, oof = [], {}
    for name, model in make_baselines().items():
        cv = cv_sklearn(model, mats)
        oof[name] = cv["oof"]
        (ap, ap_sd), (auc, auc_sd) = mean_sd(cv["ap"]), mean_sd(cv["auc"])
        rows.append({"model": name, "AP (mean)": ap, "AP (sd)": ap_sd, "ROC-AUC (mean)": auc, "ROC-AUC (sd)": auc_sd})
        results.metrics[name] = {"ap_folds": cv["ap"], "auc_folds": cv["auc"]}
    table = pd.DataFrame(rows)
    best = table.loc[table["AP (mean)"].idxmax(), "model"]
    results.metrics.update({"best_baseline": best, "prevalence_train": float(y_train.mean()),
                            "positives_per_fold": [int(fold.y_val.sum()) for fold in mats]})
    results.table(2, f"The baselines on the {N_FOLDS} validation folds (mean and sample standard deviation across "
                     "folds)", table)
    results.write()
    print(table.to_string(index=False))
    print(f"best baseline on validation: {best}")
    return oof, best

## [`code/starting_point.py`](code/starting_point.py) — step 0: the BMI-missing indicator

In [ ]:
"""Step 0 — the starting point: the EDA's pipeline, and the check its synthesis asked for (Table 1).

The EDA found BMI missing about five times more often among the positives and could not tell whether BMI is
recorded before the event the label describes. Its plan: train with and without the BMI-missing indicator, and if
the indicator carries a gain, treat it as a collection artifact and prefer the model without it. This step runs
that check on the validation folds and returns the pipeline every later step uses, for every model alike.
"""

from functools import partial

import pandas as pd

### `starting_point_run`

In [ ]:
def starting_point_run(X_train, y_train, folds):
    """Table 1: the learned baselines and the network of step 2 with and without the indicator. Returns the factory
    of the pipeline every later step uses."""
    results = Results("starting_point")
    pipelines = {"with": build_preprocessor, "without": without_indicator}
    mats = {k: fold_matrices(X_train, y_train, folds, make) for k, make in pipelines.items()}
    widths = {k: [fold.X_train.shape[1] for fold in v] for k, v in mats.items()}

    cross_validate = {name: partial(cv_sklearn, model) for name, model in make_baselines().items()
                      if name != "Majority class"}
    cross_validate["Network (step 2)"] = partial(cv_network, DEFAULT)
    rows = []
    for name, cv in cross_validate.items():
        ap = {k: cv(m)["ap"] for k, m in mats.items()}
        (with_mean, with_sd), (without_mean, without_sd) = mean_sd(ap["with"]), mean_sd(ap["without"])
        rows.append({"model": name, "AP with the indicator": with_mean, "sd (with)": with_sd,
                     "AP without it": without_mean, "sd (without)": without_sd,
                     "gain from the indicator": with_mean - without_mean})
    table = pd.DataFrame(rows)
    results.table(1, f"Validation AP with and without the BMI-missing indicator ({N_FOLDS} folds: mean and sample "
                     "sd)", table)

    gain = table.set_index("model")["gain from the indicator"]
    drop = bool(gain["Network (step 2)"] > 0)   # the EDA's rule: a gain means the indicator is suspect
    results.metrics.update({"gain": gain.to_dict(), "drop_indicator": drop, "input_widths": widths})
    results.write()
    print(table.to_string(index=False))
    print(f"drop the BMI-missing indicator: {drop}; inputs per fold {widths}")
    return without_indicator if drop else build_preprocessor

## Run — step 0

In [ ]:
make_preprocessor = starting_point_run(X_train, y_train, cv_folds)
n_inputs = make_preprocessor().fit_transform(X_train).shape[1]
mats = fold_matrices(X_train, y_train, cv_folds, make_preprocessor)
print(f"the pipeline outputs {n_inputs} columns")

In [ ]:
show_tables("starting_point")

## Run — step 1

The baselines of `code/baselines.py`, defined above.

In [ ]:
oof_baselines, best_baseline = baselines_run(mats, y_train)

In [ ]:
show_tables("baselines")

## [`code/search.py`](code/search.py) — steps 2 to 4: the network, its training and the search

In [ ]:
"""Steps 2 to 4 — the network, its training with early stopping, regularization and the configuration search.

Everything here is measured on the validation folds of the training rows; the test set is not touched.
Tables 3-6, Figures 1-2.
"""

import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Two techniques at three strengths each, against the plain network, plus one variant combining the middle
# strengths. The dropout rates run from just below the course's range for fully connected layers (0.2-0.5) to
# its top; the weight decays span two orders of magnitude, from barely acting to strong enough to underfit.
REGULARIZATION = {                       # name: (weight_decay, dropout)
    "none": (0.0, 0.0),
    "weight decay 0.1": (0.1, 0.0),
    "weight decay 1": (1.0, 0.0),
    "weight decay 10": (10.0, 0.0),
    "dropout 0.1": (0.0, 0.1),
    "dropout 0.3": (0.0, 0.3),
    "dropout 0.5": (0.0, 0.5),
    "weight decay 1 + dropout 0.3": (1.0, 0.3),
}
LEARNING_RATES = (1e-3, 3e-3)                # the course's starting point for Adam, and three times it
ARCHITECTURES = ([16], [32, 16], [64, 32])   # one narrower and one wider network around the 32-16 of step 2
NONE_COLOR = "#eb6834"
WEIGHT_DECAY_COLORS = ["#9ec5f0", "#2a78d6", "#0d3b7a"]    # light to dark: a stronger decay
DROPOUT_COLORS = ["#9be0c4", "#1baf7a", "#0b6b47"]         # light to dark: a higher rate
FOLD_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]

### `with_regularization`

In [ ]:
def with_regularization(config, name):
    """The configuration with the weight decay and dropout of the named regularization variant."""
    weight_decay, dropout = REGULARIZATION[name]
    return {**config, "weight_decay": weight_decay, "dropout": dropout}

### `regularization_name`

In [ ]:
def regularization_name(config):
    """The name of the regularization variant a configuration uses."""
    return next(name for name, (wd, p) in REGULARIZATION.items()
                if (wd, p) == (config["weight_decay"], config["dropout"]))

### `layers`

In [ ]:
def layers(config):
    """The hidden widths as written in the tables, e.g. "32-16"."""
    return "-".join(map(str, config["hidden"]))

### `summary_row`

In [ ]:
def summary_row(cv):
    """Mean and sample sd of the validation AP, mean ROC-AUC and median restored epoch of a cross-validation."""
    (ap, ap_sd), auc = mean_sd(cv["ap"]), np.mean(cv["auc"])
    return {"AP (mean)": ap, "AP (sd)": ap_sd, "ROC-AUC (mean)": auc,
            "best epoch (median)": int(np.median(cv["best_epoch"]))}

### `architecture_table`

In [ ]:
def architecture_table(config, n_inputs):
    """Layer by layer: shapes, activation, initialization and parameters."""
    rows, width = [], n_inputs
    for i, units in enumerate(config["hidden"], start=1):
        rows.append({"layer": f"hidden {i}", "shape": f"{width} → {units}", "activation": "ReLU",
                     "initialization": "He (Kaiming) normal", "parameters": width * units + units})
        width = units
    rows.append({"layer": "output", "shape": f"{width} → 1", "activation": "none (logit)",
                 "initialization": "Glorot (Xavier) normal", "parameters": width + 1})
    table = pd.DataFrame(rows)
    assert table["parameters"].sum() == MLP(n_inputs, config["hidden"]).n_parameters()
    return table

### `plot_training_curves`

In [ ]:
def plot_training_curves(cv):
    """Figure 1: training and validation loss, and validation AP, per epoch for every fold."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for k, history in enumerate(cv["histories"]):
        epochs = np.arange(1, len(history["val_loss"]) + 1)
        color, best = FOLD_COLORS[k], cv["best_epoch"][k]
        axes[0].plot(epochs, history["train_loss"], color=color, lw=1, ls="--")
        axes[0].plot(epochs, history["val_loss"], color=color, lw=1.4, label=f"fold {k + 1}")
        axes[0].plot(best, history["val_loss"][best - 1], "o", color=color, ms=5)
        axes[1].plot(epochs, history["val_ap"], color=color, lw=1.4, label=f"fold {k + 1}")
        axes[1].plot(best, history["val_ap"][best - 1], "o", color=color, ms=5)
    axes[0].plot([], [], color=NEUTRAL, ls="--", label="training loss")
    axes[0].plot([], [], "o", color=NEUTRAL, label="best epoch (restored)")
    axes[0].set(title="Loss per epoch: validation (solid), training (dashed)", xlabel="epoch",
                ylabel="weighted binary cross-entropy")
    axes[1].plot([], [], "o", color=NEUTRAL, label="best epoch (restored)")
    axes[1].set(title="Validation average precision per epoch", xlabel="epoch", ylabel="average precision")
    axes[0].legend(ncol=2)
    axes[1].legend(ncol=2)
    save_figure(fig, 1, "The network of step 2 trained with early stopping, one curve per validation fold")

### `plot_regularization`

In [ ]:
def plot_regularization(runs):
    """Figure 2: fold 1, training (dashed) and validation (solid) loss, each technique at its three strengths next to
    the network without regularization."""
    panels = {"Weight decay": dict(zip([n for n in REGULARIZATION if n.startswith("weight decay") and "+" not in n],
                                       WEIGHT_DECAY_COLORS)),
              "Dropout": dict(zip([n for n in REGULARIZATION if n.startswith("dropout")], DROPOUT_COLORS))}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
    for ax, (technique, colors) in zip(axes, panels.items()):
        for name, color in {"none": NONE_COLOR, **colors}.items():
            history, best = runs[name]["histories"][0], runs[name]["best_epoch"][0]
            epochs = np.arange(1, len(history["val_loss"]) + 1)
            ax.plot(epochs, history["train_loss"], color=color, lw=1, ls="--")
            ax.plot(epochs, history["val_loss"], color=color, lw=1.6, label=name)
            ax.plot(best, history["val_loss"][best - 1], "o", color=color, ms=5)
        ax.plot([], [], color=NEUTRAL, ls="--", label="training loss (dashed)")
        ax.plot([], [], "o", color=NEUTRAL, label="best epoch (restored)")
        ax.set(title=f"{technique}: validation (solid) and training (dashed) loss, fold 1", xlabel="epoch")
        ax.legend(title="regularization")
    axes[0].set_ylabel("weighted binary cross-entropy")
    axes[0].set_ylim(top=1.6)   # headroom so the legends cover no curve (the y axis is shared)
    save_figure(fig, 2, f"The network of step 2 ({layers(DEFAULT)}, lr {DEFAULT['lr']:g}) with each technique at "
                        "three strengths")

### `search_run`

In [ ]:
def search_run(y_train, mats, n_inputs):
    """Steps 2 to 4 on the fold matrices `mats` (Tables 3-6, Figures 1-2). Returns the chosen configuration, its
    out-of-fold probabilities and the number of epochs of the final retraining."""
    results = Results("search")

    # --- Step 2: the network ---------------------------------------------------------------------
    results.table(3, f"The network of step 2, layer by layer ({n_inputs} inputs)",
                  architecture_table(DEFAULT, n_inputs))
    n_params = MLP(n_inputs, DEFAULT["hidden"]).n_parameters()
    results.metrics.update({
        "n_inputs": n_inputs, "fold_inputs": [fold.X_train.shape[1] for fold in mats], "default_params": n_params,
        "train_rows": len(y_train), "fold_train_rows": [len(fold.y_train) for fold in mats], "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
        "pos_weight_folds": [float((fold.y_train == 0).sum() / fold.y_train.sum()) for fold in mats],
    })

    # --- Step 3: the network trained with early stopping -----------------------------------------
    default_cv = cv_network(DEFAULT, mats)
    rows = []
    for k, history in enumerate(default_cv["histories"]):
        best = default_cv["best_epoch"][k]
        rows.append({"fold": k + 1, "validation positives": int(mats[k].y_val.sum()),
                     "epochs run": len(history["val_loss"]), "best epoch": best,
                     "training loss": history["train_loss"][best - 1], "validation loss": history["val_loss"][best - 1],
                     "validation AP": default_cv["ap"][k]})
    results.table(4, "Early stopping of the network of step 2 on each fold (losses and AP at the restored epoch)",
                  pd.DataFrame(rows))
    results.metrics["default"] = {
        "ap": default_cv["ap"], "auc": default_cv["auc"], "best_epoch": default_cv["best_epoch"],
        "epochs_run": [len(h["val_loss"]) for h in default_cv["histories"]],
        "train_loss_last": [h["train_loss"][-1] for h in default_cv["histories"]],
        "val_loss_last": [h["val_loss"][-1] for h in default_cv["histories"]],
        "val_loss_best": [h["val_loss"][b - 1] for h, b in zip(default_cv["histories"], default_cv["best_epoch"])],
        # the validation AP over the whole run: its maximum, the epoch of that maximum, and its last value, to compare
        # what the loss and the ranking say about when to stop
        "val_ap_max": [max(h["val_ap"]) for h in default_cv["histories"]],
        "val_ap_max_epoch": [int(np.argmax(h["val_ap"])) + 1 for h in default_cv["histories"]],
        "val_ap_last": [h["val_ap"][-1] for h in default_cv["histories"]],
    }
    plot_training_curves(default_cv)

    # --- Step 4: regularization, then the search -------------------------------------------------
    cache = {}

    def cv(config):
        key = (config["lr"], tuple(config["hidden"]), config["weight_decay"], config["dropout"])
        if key not in cache:
            cache[key] = default_cv if config == DEFAULT else cv_network(config, mats)
        return cache[key]

    reg_runs = {name: cv(with_regularization(DEFAULT, name)) for name in REGULARIZATION}
    rows = []
    for name, run_ in reg_runs.items():
        gap = np.mean([h["val_loss"][b - 1] - h["train_loss"][b - 1]
                       for h, b in zip(run_["histories"], run_["best_epoch"])])
        rows.append({"regularization": name, **summary_row(run_), "validation − training loss": gap})
    results.table(5, f"The network of step 2 ({layers(DEFAULT)}, lr {DEFAULT['lr']:g}) with each regularization "
                     f"({N_FOLDS} validation folds; the loss gap is at the restored epoch)", pd.DataFrame(rows))
    plot_regularization(reg_runs)

    configs = [with_regularization({"hidden": list(h), "lr": lr, "dropout": 0.0, "weight_decay": 0.0}, reg)
               for lr, h, reg in itertools.product(LEARNING_RATES, ARCHITECTURES, REGULARIZATION)]
    rows = []
    for config in configs:
        rows.append({"learning rate": config["lr"], "hidden layers": layers(config),
                     "regularization": regularization_name(config),
                     "parameters": MLP(n_inputs, config["hidden"]).n_parameters(), **summary_row(cv(config))})
    # sorted by mean AP; the index keeps each row's position in `configs`
    search = pd.DataFrame(rows).sort_values("AP (mean)", ascending=False, kind="stable")
    search.insert(0, "rank", np.arange(1, len(search) + 1))

    # Selection, on validation only: the differences at the top are within the fold-to-fold noise, so the one-
    # standard-error rule picks the smallest network whose mean AP is within one standard error of the best one
    # (among networks of the same size, the one with the highest mean AP).
    best = search.iloc[0]
    threshold = best["AP (mean)"] - best["AP (sd)"] / np.sqrt(N_FOLDS)
    eligible = search[search["AP (mean)"] >= threshold]
    pick = eligible.sort_values(["parameters", "AP (mean)"], ascending=[True, False], kind="stable").iloc[0]
    search["within one SE"] = np.where(search["AP (mean)"] >= threshold, "yes", "")
    search["chosen"] = np.where(search["rank"] == pick["rank"], "◀", "")
    results.table(6, f"The search: {len(configs)} configurations on the {N_FOLDS} validation folds, sorted by mean "
                     "AP", search, floatfmt={"learning rate": "g"})
    chosen = configs[pick.name]
    chosen_cv = cv(chosen)

    results.metrics.update({
        "n_configs": len(configs), "best_rank1": best.to_dict(), "one_se_threshold": threshold,
        "n_within_one_se": int(len(eligible)), "chosen": chosen, "chosen_rank": int(pick["rank"]),
        "chosen_params": int(pick["parameters"]), "chosen_label": f"{layers(chosen)}, {pick['regularization']}",
        "chosen_cv": {"ap": chosen_cv["ap"], "auc": chosen_cv["auc"], "best_epoch": chosen_cv["best_epoch"]},
        "final_epochs": int(np.median(chosen_cv["best_epoch"])),
    })
    results.write()
    print(search.head(10).to_string(index=False))
    print(f"chosen: rank {int(pick['rank'])}, {results.metrics['chosen_label']}, lr {chosen['lr']:g}; "
          f"final epochs {results.metrics['final_epochs']}")
    return {"config": chosen, "oof": chosen_cv["oof"], "epochs": results.metrics["final_epochs"]}

## Run — steps 2 to 4

The regularization comparison and the 48-configuration search; this is the slow cell (a few minutes on a
laptop CPU).

In [ ]:
chosen = search_run(y_train, mats, n_inputs)
show_figures(1, 2)

In [ ]:
show_tables("search")

## [`code/evaluate.py`](code/evaluate.py) — step 5: the threshold, the test set and the seeds

In [ ]:
"""Step 5 — the test evaluation, done once.

Everything that is decided here is decided on validation first: each model's decision threshold comes from its
out-of-fold validation probabilities and the cost of the errors fixed in step 0. Only then are the chosen network
(retrained on all training rows, with each seed) and the baselines (refitted on all training rows) applied to the
test rows, once. Tables 7-10, Figures 3-5.
"""

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, precision_recall_curve, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)


COST_SENSITIVITY = (10, COST_MISS, 50)   # the threshold under other costs of a miss, on validation only
N_BOOTSTRAP = 2000                        # resamples of the test rows for the paired bootstrap
LEARNED_BASELINES = ("Logistic regression", "Gradient boosting")
COST_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]   # one per cost of COST_SENSITIVITY; the chosen one in orange

### `choose_threshold`

In [ ]:
def choose_threshold(y, p, cost_miss):
    """The probability threshold (flag when p >= t) with the lowest total cost cost_miss * FN + FP on (y, p).
    Candidates are the observed probabilities and +inf (flag nobody); ties go to the highest threshold."""
    candidates = np.append(np.unique(p), np.inf)
    costs = np.array([cost_miss * np.sum((p < t) & (y == 1)) + np.sum((p >= t) & (y == 0)) for t in candidates])
    best = np.flatnonzero(costs == costs.min()).max()
    return float(candidates[best]), costs, candidates

### `decision_metrics`

In [ ]:
def decision_metrics(y, p, threshold, cost_miss=COST_MISS):
    """The confusion counts, precision, recall and F1 of both classes, macro F1 and the cost per patient."""
    flag = p >= threshold
    tn, fp = int(np.sum(~flag & (y == 0))), int(np.sum(flag & (y == 0)))
    fn, tp = int(np.sum(~flag & (y == 1))), int(np.sum(flag & (y == 1)))
    precision, recall, f1, _ = precision_recall_fscore_support(y, flag.astype(int), labels=[0, 1], zero_division=0)
    return {"TN": tn, "FP": fp, "FN": fn, "TP": tp, "precision": precision.tolist(), "recall": recall.tolist(),
            "f1": f1.tolist(), "macro_f1": float(np.mean(f1)), "flagged": int(flag.sum()),
            "cost_per_patient": (cost_miss * fn + fp) / len(y)}

### `plot_threshold`

In [ ]:
def plot_threshold(y, p, chosen):
    """Figure 3: validation cost per patient against the threshold, for three costs of a miss."""
    fig, ax = plt.subplots(figsize=(9, 4.6))
    for cost, color, lw in zip(COST_SENSITIVITY, COST_COLORS, [1.2, 2.2, 1.2]):
        t, costs, candidates = choose_threshold(y, p, cost)
        finite = np.isfinite(candidates)
        ax.plot(candidates[finite], costs[finite] / len(y), color=color, lw=lw,
                label=f"a miss costs {cost} false alarms (best threshold {t:.3f})")
        ax.plot(t, costs[candidates == t][0] / len(y), "o", color=color, ms=6)
    ax.axvline(chosen, color=NEUTRAL, ls=":", lw=1, label=f"chosen threshold {chosen:.3f} (cost {COST_MISS})")
    ax.set(title="Out-of-fold validation probabilities of the chosen network",
           xlabel="threshold on the predicted probability", ylabel="cost per patient (false alarm = 1)", xlim=(0, 1))
    ax.legend()
    save_figure(fig, 3, "Choosing the decision threshold on validation from the cost of the errors")

### `plot_test_curves`

In [ ]:
def plot_test_curves(y, probabilities, metrics, labels):
    """Figure 4: precision-recall and ROC curves on the test set, with each model's operating point."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    for name, p in probabilities.items():
        color, m, label = MODEL_COLORS[name], metrics[name], labels.get(name, name)
        precision, recall, _ = precision_recall_curve(y, p)
        fpr, tpr, _ = roc_curve(y, p)
        axes[0].step(recall, precision, where="post", color=color, lw=1.6, label=f"{label} (AP {m['ap']:.3f})")
        axes[1].plot(fpr, tpr, color=color, lw=1.6, label=f"{label} (ROC-AUC {m['auc']:.3f})")
        if m["TP"] + m["FP"] > 0:   # precision is undefined for a model that flags nobody
            axes[0].plot(m["recall"][1], m["precision"][1], "o", color=color, ms=6)
        axes[1].plot(m["FP"] / (m["FP"] + m["TN"]), m["recall"][1], "o", color=color, ms=6)
    # no prevalence line or diagonal is drawn: the majority class, which gives every patient the same score, traces both
    axes[0].set(title="Precision–recall (dots: the validation threshold)", xlabel="recall (strokes found)",
                ylabel="precision (alarms that are strokes)", xlim=(0, 1), ylim=(0, 1))
    axes[1].set(title="ROC (dots: the validation threshold)", xlabel="false-positive rate",
                ylabel="true-positive rate")
    axes[0].legend(fontsize=7.5)
    axes[1].legend(fontsize=7.5, loc="lower right")
    save_figure(fig, 4, "The network and the baselines on the test set")

### `plot_confusion`

In [ ]:
def plot_confusion(matrices):
    """Figure 5: confusion matrices on the test set at the validation thresholds."""
    fig, axes = plt.subplots(1, len(matrices), figsize=(4.4 * len(matrices), 4.2))
    for ax, (name, m) in zip(axes, matrices.items()):
        cm = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])
        image = ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.sum())   # the same color scale in every panel
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=13,
                        color="white" if cm[i, j] > cm.sum() / 2 else "black")
        ax.set_xticks([0, 1], ["no stroke", "stroke"])
        ax.set_yticks([0, 1], ["no stroke", "stroke"])
        ax.set(title=f"{name}\nthreshold {m['threshold']:.3f}", xlabel="predicted", ylabel="actual")
        ax.grid(False)
        fig.colorbar(image, ax=ax, fraction=0.046, label="test patients")
    save_figure(fig, 5, "Confusion matrices on the test set at the validation thresholds")

### `paired_bootstrap`

In [ ]:
def paired_bootstrap(y, network_seeds, baselines):
    """Resample the test rows with replacement, the same rows for every model; on each resample, the AP of the network
    (the seed-42 model and the mean over the seeds) and its gap to each baseline. A resample without a positive has
    no AP and is skipped."""
    rng = np.random.default_rng(SEED)
    one_seed = f"seed {SEED}"
    ap = {one_seed: [], "mean of the seeds": []}
    diffs = {(net, name): [] for net in ap for name in baselines}
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].sum() == 0:
            continue
        seeds = [average_precision_score(y[idx], p[idx]) for p in network_seeds.values()]
        nets = {one_seed: seeds[list(network_seeds).index(SEED)], "mean of the seeds": float(np.mean(seeds))}
        for net, value in nets.items():
            ap[net].append(value)
            for name, p in baselines.items():
                diffs[(net, name)].append(value - average_precision_score(y[idx], p[idx]))
    return {"n": len(ap[one_seed]),
            "network_ap_ci": {k: np.percentile(v, [2.5, 97.5]).tolist() for k, v in ap.items()},
            "diff": {f"{net} - {name}": {"ci": np.percentile(d, [2.5, 97.5]).tolist(),
                                         "share_positive": float(np.mean(np.array(d) > 0))}
                     for (net, name), d in diffs.items()}}

### `evaluate_run`

In [ ]:
def evaluate_run(X_train, X_test, y_train, y_test, make_preprocessor, chosen, oof_baselines, best_baseline):
    """Step 5 (Tables 7-10, Figures 3-5). Returns what step 6 analyzes: the seed-42 network, the fitted pipeline, the
    test probabilities of every model and the network's threshold."""
    results = Results("evaluation")

    # --- On validation: the thresholds -----------------------------------------------------------
    oof = {"Network": chosen["oof"], **oof_baselines}
    # The same rule for every model. The majority class gives every patient the same score, so its only choices are
    # flagging everyone or no one; the rule picks the cheaper of the two on validation.
    thresholds = {name: choose_threshold(y_train, p, COST_MISS)[0] for name, p in oof.items()}
    rows = []
    for cost in COST_SENSITIVITY:
        t = choose_threshold(y_train, chosen["oof"], cost)[0]
        m = decision_metrics(y_train, chosen["oof"], t, cost)
        rows.append({"cost of a miss": cost, "threshold": t, "recall (1)": m["recall"][1],
                     "precision (1)": m["precision"][1], "flagged (% of patients)": 100 * m["flagged"] / len(y_train),
                     "cost per patient": m["cost_per_patient"]})
    results.table(7, "The chosen network's threshold on the out-of-fold validation probabilities, for three costs "
                     "of a miss", pd.DataFrame(rows),
                  floatfmt={"cost of a miss": "d", "flagged (% of patients)": ".2f"})
    plot_threshold(y_train, chosen["oof"], thresholds["Network"])
    # pos_weight multiplies the odds the network learns by w = negatives / positives; dividing the odds of the threshold
    # by w gives the probability it stands for once that factor is undone, to compare with p* = 1 / (1 + COST_MISS)
    w = (y_train == 0).sum() / (y_train == 1).sum()
    t = thresholds["Network"]
    results.metrics.update({
        "thresholds": {k: (v if np.isfinite(v) else None) for k, v in thresholds.items()}, "pos_weight": float(w),
        "threshold_corrected": t / (t + (1 - t) * w), "p_star": 1 / (1 + COST_MISS),
        "validation_decisions": {name: decision_metrics(y_train, p, thresholds[name]) for name, p in oof.items()},
    })

    # --- Once: the final models on the test rows -------------------------------------------------
    preprocessor = make_preprocessor().fit(X_train)
    X_train_t, X_test_t = preprocessor.transform(X_train), preprocessor.transform(X_test)
    # The seeds change only the initial weights and the order of the mini-batches: the rows, the configuration,
    # the number of epochs and the threshold are the same for all five.
    seed_models = {seed: train(X_train_t, y_train, chosen["config"], seed, epochs=chosen["epochs"])[0]
                   for seed in SEEDS}
    seed_probs = {seed: predict_proba(model, X_test_t) for seed, model in seed_models.items()}
    probabilities = {"Network": seed_probs[SEED]}
    for name, model in make_baselines().items():
        probabilities[name] = model.fit(X_train_t, y_train).predict_proba(X_test_t)[:, 1]

    metrics = {}
    for name, p in probabilities.items():
        metrics[name] = decision_metrics(y_test, p, thresholds[name])
        metrics[name].update({"ap": average_precision_score(y_test, p), "auc": roc_auc_score(y_test, p),
                              "threshold": thresholds[name] if np.isfinite(thresholds[name]) else None})
    per_seed = {seed: {**decision_metrics(y_test, p, thresholds["Network"]), "ap": average_precision_score(y_test, p),
                       "auc": roc_auc_score(y_test, p)} for seed, p in seed_probs.items()}

    def mean_over_seeds(key, index=None):
        return float(np.mean([m[key] if index is None else m[key][index] for m in per_seed.values()]))

    labels = {"Network": f"Network (seed {SEED})"}
    rows, per_class = [], []
    for name, m in metrics.items():
        rows.append({"model": labels.get(name, name), "AP": m["ap"], "ROC-AUC": m["auc"],
                     "threshold (validation)": m["threshold"] if m["threshold"] is not None else np.nan,
                     "TN": m["TN"], "FP": m["FP"], "FN": m["FN"], "TP": m["TP"],
                     "cost per patient": m["cost_per_patient"]})
        per_class.append({"model": labels.get(name, name), "precision (0)": m["precision"][0],
                          "recall (0)": m["recall"][0],
                          "F1 (0)": m["f1"][0], "precision (1)": m["precision"][1], "recall (1)": m["recall"][1],
                          "F1 (1)": m["f1"][1], "macro F1": m["macro_f1"]})
        if name == "Network":   # the same row averaged over the seeds, right below the seed-42 one
            rows.append({"model": f"Network (mean of {len(SEEDS)} seeds)", "AP": mean_over_seeds("ap"),
                         "ROC-AUC": mean_over_seeds("auc"), "threshold (validation)": thresholds["Network"],
                         **{k: mean_over_seeds(k) for k in ("TN", "FP", "FN", "TP")},
                         "cost per patient": mean_over_seeds("cost_per_patient")})
            per_class.append({"model": f"Network (mean of {len(SEEDS)} seeds)",
                              **{f"{metric} ({c})": mean_over_seeds(key, c) for c in (0, 1)
                                 for metric, key in (("precision", "precision"), ("recall", "recall"), ("F1", "f1"))},
                              "macro F1": mean_over_seeds("macro_f1")})
    counts = {k: ".1f" for k in ("TN", "FP", "FN", "TP")}   # the seed mean is not a whole number of patients
    # dtype=object keeps each model's whole counts as integers next to the seed means
    results.table(8, "The test set, used once: AP, ROC-AUC and the decisions at each model's validation threshold",
                  pd.DataFrame(rows, dtype=object), floatfmt=counts)
    columns = ["model", "precision (0)", "recall (0)", "F1 (0)", "precision (1)", "recall (1)", "F1 (1)", "macro F1"]
    results.table(9, "Precision, recall and F1 per class on the test set (0 = no stroke, 1 = stroke)",
                  pd.DataFrame(per_class)[columns])

    seeds = pd.DataFrame([{"seed": seed, "AP": m["ap"], "ROC-AUC": m["auc"], "recall (1)": m["recall"][1],
                           "false alarms": m["FP"], "cost per patient": m["cost_per_patient"]}
                          for seed, m in per_seed.items()])
    (ap_mean, ap_sd), (auc_mean, auc_sd) = mean_sd(seeds["AP"]), mean_sd(seeds["ROC-AUC"])
    results.table(10, f"The chosen network retrained with {len(SEEDS)} seeds, on the test set", seeds,
                  floatfmt={"seed": "d"})

    # --- The success criterion of step 0, and how far the numbers can be trusted -----------------
    base_ap = metrics[best_baseline]["ap"]
    gain = ap_mean / base_ap - 1
    met = bool(gain >= CRITERION_GAIN and ap_mean - base_ap > ap_sd)
    bootstrap = paired_bootstrap(y_test, seed_probs, {k: probabilities[k] for k in LEARNED_BASELINES})
    cost_rows = {"flag nobody": COST_MISS * y_test.sum() / len(y_test),
                 "flag everyone": (y_test == 0).sum() / len(y_test)}
    results.metrics.update({
        "test": metrics, "per_seed": per_seed,
        "seeds": {"ap": seeds["AP"].tolist(), "auc": seeds["ROC-AUC"].tolist(), "ap_mean": ap_mean, "ap_sd": ap_sd,
                  "auc_mean": auc_mean, "auc_sd": auc_sd},
        "best_baseline": best_baseline, "best_baseline_ap": base_ap, "criterion_gain": gain, "criterion_met": met,
        "gain_vs": {k: ap_mean / metrics[k]["ap"] - 1 for k in LEARNED_BASELINES}, "bootstrap": bootstrap,
        "trivial_costs": cost_rows, "test_rows": len(y_test), "test_positives": int(y_test.sum()),
        "epochs": chosen["epochs"],
    })
    plot_test_curves(y_test, probabilities, metrics, labels)
    plot_confusion({labels.get(name, name): metrics[name] for name in ("Network", *LEARNED_BASELINES)})
    results.write()
    print(seeds.to_string(index=False))
    print(f"network AP {ap_mean:.4f} ± {ap_sd:.4f} vs {best_baseline} {base_ap:.4f}: gain {gain:+.2%}, met: {met}")
    print(f"bootstrap: {json.dumps(bootstrap)}")
    return {"model": seed_models[SEED], "preprocessor": preprocessor, "probabilities": probabilities,
            "threshold": thresholds["Network"]}

## Run — step 5

The first and only time the test rows are read.

In [ ]:
final = evaluate_run(X_train, X_test, y_train, y_test, make_preprocessor, chosen,
                     oof_baselines, best_baseline)
show_figures(3, 4, 5)

In [ ]:
show_tables("evaluation")

## [`code/errors.py`](code/errors.py) — step 6: the errors by subgroup and permutation importance

In [ ]:
"""Step 6 — error analysis of the chosen network (seed 42) on the test set, after the single evaluation of step 5:
errors per subgroup, the worst examples and permutation importance.
Nothing here feeds back into any choice. Tables 11-13, Figures 6-7.
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score


N_REPEATS = 30            # permutations per feature
N_WORST = 5               # worst examples shown of each kind of error
GLUCOSE_VALLEY = 175      # mg/dL: the valley between the two glucose modes found by the EDA
ADULT, OLD = 18, 60       # the EDA's age bands: under 18, 18 to 59, 60 or more

### `age_bands`

In [ ]:
def age_bands(age):
    """The EDA's three age bands, as masks."""
    return {f"under {ADULT}": age < ADULT, f"{ADULT}–{OLD - 1}": (age >= ADULT) & (age < OLD),
            f"{OLD} or more": age >= OLD}

### `subgroups`

In [ ]:
def subgroups(X):
    """Subgroups taken from the EDA's findings — age bands, the clinical flags, BMI missingness, the glucose mode —
    and gender, which the EDA found unrelated to stroke, as a check that the errors do not fall on one group."""
    return {
        **{("age", band): mask for band, mask in age_bands(X["age"]).items()},
        ("hypertension", "no"): X["hypertension"] == 0, ("hypertension", "yes"): X["hypertension"] == 1,
        ("heart disease", "no"): X["heart_disease"] == 0, ("heart disease", "yes"): X["heart_disease"] == 1,
        ("BMI", "recorded"): X["bmi"].notna(), ("BMI", "missing"): X["bmi"].isna(),
        ("glucose", f"below {GLUCOSE_VALLEY}"): X["avg_glucose_level"] < GLUCOSE_VALLEY,
        ("glucose", f"{GLUCOSE_VALLEY} or more"): X["avg_glucose_level"] >= GLUCOSE_VALLEY,
        ("gender", "female"): X["gender"] == "Female", ("gender", "male"): X["gender"] == "Male",
    }

### `plot_subgroups`

In [ ]:
def plot_subgroups(table):
    """Figure 6: miss rate and false-alarm rate per subgroup."""
    labels = [f"{f}: {g}" for f, g in zip(table["feature"], table["subgroup"])]
    y = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(9, 6.2))
    ax.barh(y - 0.2, table["miss rate (%)"], height=0.4, color=CLASS_COLORS[1], label="miss rate (% of its strokes)")
    ax.barh(y + 0.2, table["false-alarm rate (%)"], height=0.4, color=CLASS_COLORS[0],
            label="false-alarm rate (% of its non-strokes)")
    ax.set_yticks(y, labels)
    ax.invert_yaxis()
    ax.set(title="Test patients, the network at its validation threshold", xlabel="rate (%)", ylabel="subgroup")
    ax.legend(loc="lower right")
    save_figure(fig, 6, "Errors of the network by subgroup")

### `plot_importance`

In [ ]:
def plot_importance(table):
    """Figure 7: permutation importance (drop in test AP) of each recorded feature."""
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    y = np.arange(len(table))
    ax.barh(y, table["AP drop (mean)"], xerr=table["AP drop (sd)"], color=CLASS_COLORS[1], capsize=3,
            label=f"mean ± sd over {N_REPEATS} permutations")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(y, table["feature"])
    ax.invert_yaxis()
    ax.set(title="Drop in test average precision when one recorded feature is shuffled", xlabel="AP drop",
           ylabel="feature (as recorded)")
    ax.legend(loc="lower right")
    save_figure(fig, 7, "Permutation importance of the network on the test set")

### `clinical_flags`

In [ ]:
def clinical_flags(X):
    """The three recorded risk markers the EDA found associated with stroke beyond age."""
    return {"hypertension": X["hypertension"] == 1, "heart disease": X["heart_disease"] == 1,
            f"glucose {GLUCOSE_VALLEY} or more": X["avg_glucose_level"] >= GLUCOSE_VALLEY}

### `errors_run`

In [ ]:
def errors_run(X_test, y_test, final):
    """Step 6 (Tables 11-13, Figures 6-7), on the seed-42 network of step 5 at its validation threshold."""
    results = Results("errors")
    p, t = final["probabilities"]["Network"], final["threshold"]
    alarm = p >= t   # the network's decisions: True flags the patient

    rows = []
    for (feature, group), mask in subgroups(X_test).items():
        mask = mask.to_numpy()
        pos, neg = int((y_test[mask] == 1).sum()), int((y_test[mask] == 0).sum())
        fn, fp = int((~alarm & mask & (y_test == 1)).sum()), int((alarm & mask & (y_test == 0)).sum())
        rows.append({"feature": feature, "subgroup": group, "patients": int(mask.sum()), "strokes": pos, "missed": fn,
                     "false alarms": fp, "miss rate (%)": 100 * fn / pos if pos else np.nan,
                     "false-alarm rate (%)": 100 * fp / neg if neg else np.nan,
                     "error rate (%)": 100 * (fn + fp) / mask.sum(),
                     "cost per patient": (COST_MISS * fn + fp) / mask.sum()})
    table = pd.DataFrame(rows)
    results.table(11, "Errors of the network by subgroup on the test set, at the validation threshold", table,
                  floatfmt={"miss rate (%)": ".2f", "false-alarm rate (%)": ".2f", "error rate (%)": ".2f"})
    worst = table.loc[table["error rate (%)"].idxmax()]
    costliest = table.loc[table["cost per patient"].idxmax()]
    plot_subgroups(table)

    # The subgroups the hypotheses below refer to: the adult age bands and the three clinical flags.
    age = X_test["age"].to_numpy()
    bands = age_bands(age)
    markers = {name: has.to_numpy() for name, has in clinical_flags(X_test).items()}
    any_marker = np.any(list(markers.values()), axis=0)
    strokes_18_59 = bands[f"{ADULT}–{OLD - 1}"] & (y_test == 1)
    old = bands[f"{OLD} or more"]
    hyper = X_test["hypertension"].to_numpy() == 1

    ids = load_raw()["id"]
    shown = ["age", "gender", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "smoking_status"]
    missed = np.flatnonzero(~alarm & (y_test == 1))
    false_alarms = np.flatnonzero(alarm & (y_test == 0))
    picks = [("missed stroke", i) for i in missed[np.argsort(p[missed])][:N_WORST]] + \
            [("false alarm", i) for i in false_alarms[np.argsort(-p[false_alarms])][:N_WORST]]
    # the id as text, so the table shows it as recorded (49669) rather than as a count (49,669)
    examples = pd.DataFrame([{"error": kind, "id": str(ids[X_test.index[i]]), **X_test.iloc[i][shown].to_dict(),
                              "probability": p[i]} for kind, i in picks])
    examples["bmi"] = examples["bmi"].map(lambda v: "missing" if pd.isna(v) else f"{v:.1f}")
    results.table(12, f"The worst errors: the {N_WORST} strokes with the lowest probability and the {N_WORST} "
                      "non-strokes with the highest", examples,
                  floatfmt={"age": "g", "avg_glucose_level": ".2f", "probability": ".4f"})

    model, preprocessor = final["model"], final["preprocessor"]
    base = average_precision_score(y_test, p)
    rng = np.random.default_rng(SEED)
    rows = []
    # Each feature is shuffled as it was recorded and the pipeline is applied after the shuffle, so a categorical
    # feature moves with all of its one-hot columns instead of having its importance split between them.
    for feature in FEATURES:
        drops = []
        for _ in range(N_REPEATS):
            X = X_test.copy()
            X[feature] = rng.permutation(X[feature].to_numpy())
            drops.append(base - average_precision_score(y_test, predict_proba(model, preprocessor.transform(X))))
        rows.append({"feature": feature, "AP drop (mean)": np.mean(drops), "AP drop (sd)": np.std(drops, ddof=1)})
    importance = pd.DataFrame(rows).sort_values("AP drop (mean)", ascending=False, kind="stable").reset_index(drop=True)
    results.table(13, f"Permutation importance on the test set ({N_REPEATS} permutations per feature; "
                      f"test AP {base:.4f})", importance)
    plot_importance(importance)

    results.metrics.update({
        "worst_subgroup": f"{worst['feature']}: {worst['subgroup']}",
        "worst_error_rate": float(worst["error rate (%)"]),
        "worst_subgroup_row": worst.to_dict(), "costliest_subgroup_row": costliest.to_dict(),
        "subgroups": table.to_dict(orient="records"),
        "strokes_18_59": int(strokes_18_59.sum()),
        "strokes_18_59_with_clinical_flag": int((strokes_18_59 & any_marker).sum()),
        "strokes_18_59_with_clinical_flag_missed": int((strokes_18_59 & any_marker & ~alarm).sum()),
        "stroke_rate_60_plus": {"hypertension": float(y_test[old & hyper].mean()),
                                "no hypertension": float(y_test[old & ~hyper].mean()),
                                "n_hypertension": int((old & hyper).sum()),
                                "n_no_hypertension": int((old & ~hyper).sum())},
        "test_strokes_hypertension": int((hyper & (y_test == 1)).sum()),
        "importance": importance.to_dict(orient="records"), "base_ap": base,
    })
    results.write()
    print(table.to_string(index=False))
    print(examples.to_string(index=False))
    print(importance.to_string(index=False))

## Run — step 6

In [ ]:
errors_run(X_test, y_test, final)
show_figures(6, 7)

In [ ]:
show_tables("errors")

## The split sizes the report quotes

The last lines of `main()`.

In [ ]:
import json

(RESULTS_DIR / "split_sizes.json").write_text(json.dumps({
    "train": len(y_train), "train_positives": int(y_train.sum()), "test": len(y_test),
    "test_positives": int(y_test.sum()), "validation_folds": [len(v) for _, v in cv_folds],
    "validation_positives": [int(y_train[v].sum()) for _, v in cv_folds]}, indent=2) + "\n")
print((RESULTS_DIR / "split_sizes.json").read_text())